In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


import os
import sys

project_pth = (os.path.join(os.getcwd(),'..','..'))

sys.path.append(project_pth)



In [0]:
from utils.Transformations import reusable

# DimUser

## AutoLoader


In [0]:
DimUser_checkpoint = "abfss://silver@sportifystorages.dfs.core.windows.net/DimUser/checkpoint"

In [0]:
df_user = spark.readStream.format("cloudFiles")\
         .option("cloudFiles.format", "parquet")\
         .option("cloudFiles.schemaLocation", DimUser_checkpoint)\
         .load("abfss://bronze@sportifystorages.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))




In [0]:
df_user_obj = reusable()


df_user = df_user_obj.dropColumns(df_user,['_rescued_data','end_date'])
df_user = df_user.dropDuplicates(['user_id'])


In [0]:
display(df_user, checkpointLocation = DimUser_checkpoint)

In [0]:
df_user.writeStream.format('delta')\
    .outputMode("append")\
    .option("checkpointLocation", DimUser_checkpoint)\
    .trigger(once=True)\
    .option("path", "abfss://silver@sportifystorages.dfs.core.windows.net/DimUser/data")\
    .toTable("sportify_cata.silver.DimUser")

# DimArtist

In [0]:
DimArtist_checkpoint = "abfss://silver@sportifystorages.dfs.core.windows.net/DimArtist/checkpoint"

In [0]:
df_art =spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", DimArtist_checkpoint)\
    .load("abfss://bronze@sportifystorages.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(["artist_id"])

In [0]:
df_art.writeStream.format('delta')\
    .outputMode("append")\
    .option("checkpointLocation", DimArtist_checkpoint)\
    .trigger(once=True)\
    .option("path","abfss://silver@sportifystorages.dfs.core.windows.net/DimArtist/data")\
    .toTable("sportify_cata.silver.DimArtist")


# DimTrack

In [0]:
DimTrack_checkpoint = "abfss://silver@sportifystorages.dfs.core.windows.net/DimTrack/checkpoint"

In [0]:
df_track = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", DimTrack_checkpoint)\
    .load("abfss://bronze@sportifystorages.dfs.core.windows.net/DimTrack")


In [0]:
df_track = df_track.withColumn("durationFlag",when(col("duration_sec")<150,"low")\
                                             .when(col("duration_sec")<300,"medium")\
                                             .otherwise("high"))

In [0]:
df_track = df_track.withColumn("track_name",regexp_replace(col("track_name"),"-"," "))

In [0]:
df_track_obj = reusable()
df_track = df_track_obj.dropColumns(df_track,["_rescued_data"])

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", DimTrack_checkpoint)\
    .trigger(once=True)\
    .option("path","abfss://silver@sportifystorages.dfs.core.windows.net/DimTrack/data")\
    .toTable("sportify_cata.silver.DimTrack")

# DimDate

In [0]:
DimDate_checkpoint = "abfss://silver@sportifystorages.dfs.core.windows.net/DimDate/checkpoint"

In [0]:
df_Date = spark.readStream.format("cloudFiles")\
     .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", DimDate_checkpoint)\
    .load("abfss://bronze@sportifystorages.dfs.core.windows.net/DimDate")


In [0]:
df_Date_obj = reusable()

df_Date = df_Date_obj.dropColumns(df_Date,["_rescued_data"])

In [0]:
df_Date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", DimDate_checkpoint)\
    .trigger(once=True)\
    .option("path", "abfss://silver@sportifystorages.dfs.core.windows.net/DimDate/data")\
    .toTable("sportify_cata.silver.DimDate")

# FactStream

In [0]:
FactStream_checkpoint = "abfss://silver@sportifystorages.dfs.core.windows.net/FactStream/checkpoint"

In [0]:
df_Fact = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", FactStream_checkpoint)\
    .load("abfss://bronze@sportifystorages.dfs.core.windows.net/FactStream")

In [0]:
df_Fact = reusable().dropColumns(df_Fact,["_rescued_data"])

In [0]:
df_Fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", FactStream_checkpoint)\
    .trigger(once=True)\
    .option("path", "abfss://silver@sportifystorages.dfs.core.windows.net/FactStream/data")\
    .toTable("sportify_cata.silver.FactStream")